# cositos — live widget in a clojupyter (Clojure Jupyter) kernel

This is the Clojure twin of `python_counter.ipynb` / `julia_counter.ipynb`. It builds a
**cositos** widget in Clojure and displays it over a real Jupyter **comm** channel via
clojupyter — the exact same counter ESM as the other notebooks, no new JavaScript.

**How this works, and why it's different from the Python/Julia notebooks.** clojupyter
ships no user-facing API to *open* a comm from Clojure code (`probe/README.md`,
"clojupyter comm surface"). `cositos.clojupyter-transport` (dev-only, not part of the
certified `cositos.core`) reaches into `clojupyter.state/current-context` — an internal,
version-coupled implementation detail, not a public/upstream API — confirmed live in
`cositos-059.9`. It supports the `update` (state-sync) round trip this counter needs, but
**not** binary buffers or `custom` messages (see the namespace docstring for the exact
reasons). If you need those, or want to stay off internal APIs entirely, use **Clay**
instead (`docs/hosts.md`) — it's the recommended Clojure host for cositos widgets.

**Prerequisites** (a live kernel is required — a static `.ipynb` will not render widgets):

- A clojupyter kernel with `clojupyter/clojupyter` on its classpath (any recent 0.4.x;
  confirmed against 0.4.332). `cositos.core` and `cositos.clojupyter-transport` do **not**
  need to be pre-installed on the kernel's classpath — the first code cell below adds
  them at runtime via `pomegranate` (bundled inside clojupyter), exactly as
  `tests/test_e2e_clojure.py` does automatically for CI.
- `anywidget` installed in the **frontend** (JupyterLab/Notebook) so the `anywidget`
  model/view module resolves — cositos emits `_model_module="anywidget"`.

In [ ]:
; Add this repo's clojure/src (the certified protocol core) and clojure/dev (the
; clojupyter transport adapter) to the kernel's classpath at runtime. Adjust the path
; if this notebook is opened from somewhere other than the repo root.
(require '[cemerick.pomegranate :as pom])
(def repo-root ".")
(pom/add-classpath (str repo-root "/clojure/src"))
(pom/add-classpath (str repo-root "/clojure/dev"))
(require '[cositos.clojupyter-transport :as tx]
         '[clojupyter.display :as display])

In [ ]:
; An anywidget-style ESM widget: a counter button. Identical to the Python/Julia notebooks.
(def esm
  "export default {
  render({ model, el }) {
    const btn = document.createElement(\"button\");
    const out = document.createElement(\"span\");
    const paint = () => { out.textContent = \" count = \" + model.get(\"count\"); };
    btn.textContent = \"increment\";
    btn.addEventListener(\"click\", () => {
      model.set(\"count\", model.get(\"count\") + 1);
      model.save_changes();
    });
    model.on(\"change:count\", paint);
    paint();
    el.appendChild(btn); el.appendChild(out);
  }
}"
)

In [ ]:
; The host owns the state; cositos is the protocol glue (no observer magic). open!
; sends comm_open (via the current-context crack) and returns a ClojupyterTransport.
(def state (atom {"_esm" esm "count" 0}))
(def transport (tx/open! @state))
; Inbound updates from the frontend (a click -> model.save_changes()) land here.
(tx/on-update! transport (fn [new-state] (reset! state new-state)))
nil

clojupyter has no `_repr_mimebundle_`-style auto-display hook for arbitrary types
(unlike Python/Julia's cositos ports, which register one), so rendering the widget-view
is one explicit call: `clojupyter.display/render-mime` as a cell's last expression.

In [ ]:
(display/render-mime tx/widget-view-mimetype (tx/widget-view transport))

Click **increment** in the rendered widget: the frontend calls `save_changes()`, which
sends an `update` back over the comm; `on-update!`'s handler merges it into `state`.
Read it back from the kernel:

In [ ]:
(get @state "count")  ; reflects clicks made in the frontend